# FAISS Vector Database

FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It is commonly used in semantic search, recommendation systems, image retrieval, and Retrieval-Augmented Generation (RAG) applications.

## How FAISS Works

1. Convert text, images, or other data into numerical vectors using an embedding model.
2. Store the vectors in a FAISS index.
3. Convert a query into a vector.
4. Search the index for the most similar vectors.
5. Retrieve the original data associated with the matching vectors.

## Advantages

- Fast similarity search
- Supports large collections of vectors
- Provides multiple indexing algorithms
- Supports GPU acceleration
- Works well with machine learning and NLP workflows

## Important Note

FAISS is primarily a vector search library, not a complete database. It stores vectors and performs similarity searches, but metadata and original documents are usually managed separately or with a vector database built on top of FAISS.

## Common Similarity Metrics

- **Euclidean distance**: Measures the direct distance between vectors.
- **Cosine similarity**: Measures the angle between vectors.
- **Inner product**: Often used for ranking vector similarity.

FAISS is especially useful for building efficient semantic search and RAG pipelines.

In [1]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("00_data/attention.pdf")
docs = loader.load()
print(f"Number of pages loaded: {len(docs)}")
docs[0]

/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_10619/3750619610.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/bk/brew-global-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of pages loaded: 15


Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '00_data/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nl

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)
print(f"Number of chunks: {len(split_docs)}")

Number of chunks: 52


In [ ]:

from langchain_huggingface import HuggingFaceEmbeddings

#create embedding model 

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7154.67it/s]


In [6]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(split_docs, embeddings)

print(f"Number of vectors in the FAISS database: {vector_db.index.ntotal}")

Number of vectors in the FAISS database: 52


In [8]:
query = "What is the attention mechanism?"
results = vector_db.similarity_search(query, k=3)
print("Results:")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])
    print()


Results:
--- Result 1 (page 2) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

--- Result 2 (page 12) ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
h

--- Result 3 (page 2) ---
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder in



## Using the Vector Store as a Retriever

`similarity_search` above is a FAISS-specific method. A **retriever** wraps the vector store in LangChain's standard `Runnable` interface, so the same object can be dropped into a chain or a RAG pipeline regardless of which vector store is behind it.

The search behaviour is configured once, when the retriever is created:

- `search_type="similarity"` (default) returns the `k` nearest chunks.
- `search_type="mmr"` uses Maximal Marginal Relevance to trade some similarity for diversity, which helps when the top hits are near-duplicates of each other.
- `search_type="similarity_score_threshold"` drops anything below a relevance score, so a weak query can return fewer than `k` chunks - or none at all.

### 1. `similarity` (the default)

Returns the `k` chunks whose embeddings sit closest to the query embedding - the same ranking as the `similarity_search` call above, just reached through the retriever interface.

In [9]:
# Wrap the FAISS store in the standard retriever interface
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

query = "What is the attention mechanism?"
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Document {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])
    print()

Retrieved 3 documents
--- Document 1 (page 2) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

--- Document 2 (page 12) ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
h

--- Document 3 (page 2) ---
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder in



### 2. `mmr` (Maximal Marginal Relevance)

MMR fetches a larger candidate pool (`fetch_k`), then greedily picks `k` chunks that are relevant to the query *and* unlike the chunks already picked. It is worth using when the top hits are near-duplicates of each other, which wastes context window in a RAG prompt. Compare the pages it returns against the plain similarity results above.

In [10]:
# MMR re-ranks a larger candidate pool (fetch_k) down to k diverse results
# lambda_mult: 1.0 = maximum relevance, 0.0 = maximum diversity
mmr_retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 20, "lambda_mult": 0.5},
)

for i, doc in enumerate(mmr_retriever.invoke(query)):
    print(f"--- MMR result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])
    print()

--- MMR result 1 (page 2) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

--- MMR result 2 (page 14) ---
Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
Input-Input Layer5
The
Law


--- MMR result 3 (page 4) ---
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in 



### 3. `similarity_score_threshold`

Drops any chunk whose **relevance score** falls below `score_threshold`, so a weak query can return fewer than `k` chunks - or none at all. This is what stops an off-topic question from stuffing irrelevant context into a RAG prompt.

Relevance scores are not on a universal scale. LangChain converts the store's raw distance into a 0-1 score, and the conversion depends on the distance metric and on whether the embeddings are unit-normalised. FAISS defaults to L2 distance here, and with `all-MiniLM-L6-v2` the scores land roughly between -0.3 and 0.5 - so a threshold of 0.8 would silently return nothing at all. Always look at the real scores before choosing a threshold.

In [11]:
# Inspect the real score range before picking a threshold
off_topic_query = "How do I bake sourdough bread?"

for q in (query, off_topic_query):
    scored = vector_db.similarity_search_with_relevance_scores(q, k=5)
    print(f"{q!r}")
    print("   scores:", [round(float(score), 3) for _, score in scored])

'What is the attention mechanism?'
   scores: [0.463, 0.408, 0.301, 0.3, 0.259]
'How do I bake sourdough bread?'
   scores: [-0.238, -0.25, -0.268, -0.3, -0.306]


/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_10619/104453055.py:5: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='0487df71-3d8b-4ab7-9f4e-c0c33acb078b', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '00_data/attention.pdf', 'total_pages': 15, 'page': 8, 'page_label': '9'}, page_content='Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base\nmodel. All metrics are on the English-to-German translation development set, newstest2013. Listed\nperplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to\nper-word perplexities.\nN d model dff h d k dv Pdrop ϵls

In [12]:
threshold_retriever = vector_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.25},
)

# The off-topic query returns nothing, and LangChain logs a warning saying so
for q in (query, off_topic_query):
    docs = threshold_retriever.invoke(q)
    print(f"{q!r} -> {len(docs)} document(s) above the threshold")
    for doc in docs:
        print(f"   page {doc.metadata.get('page')}: {doc.page_content[:120]}...")
    print()

/Users/bk/brew-global-venv/lib/python3.14/site-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='0487df71-3d8b-4ab7-9f4e-c0c33acb078b', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '00_data/attention.pdf', 'total_pages': 15, 'page': 8, 'page_label': '9'}, page_content='Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base\nmodel. All metrics are on the English-to-German translation development set, newstest2013. Listed\nperplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to\nper-word perplexities.\nN d model dff h

'What is the attention mechanism?' -> 3 document(s) above the threshold
   page 2: 3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where...
   page 12: Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new...
   page 2: itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce ...

'How do I bake sourdough bread?' -> 0 document(s) above the threshold



### 4. Metadata filtering

A filter is not a search type - it is a constraint layered on top of any of them. FAISS differs from a real database here: the index stores only vectors, so LangChain fetches `fetch_k` candidates first and **then** discards the ones that fail the filter. A narrow filter can therefore return fewer than `k` results, and the fix is to raise `fetch_k`.

FAISS accepts two filter forms:

- a **dict**, matched against the chunk metadata (a list value means "any of these").
- a **callable** taking the metadata dict and returning a bool - arbitrary Python, which no server-side filter can do.

In [13]:
# Dict filter: only chunks from page 2
for doc in vector_db.similarity_search(query, k=3, filter={"page": 2}):
    print(f"page {doc.metadata['page']}: {doc.page_content[:90]}...")
print()

# A list value means "any of these"
print("pages 1 or 3 :", [d.metadata["page"] for d in vector_db.similarity_search(query, k=3, filter={"page": [1, 3]})])

# Callable filter: arbitrary Python over the metadata dict.
# fetch_k is raised because filtering happens after the candidates come back.
print("pages >= 5   :", [d.metadata["page"] for d in vector_db.similarity_search(
    query, k=3, filter=lambda m: m.get("page", 0) >= 5, fetch_k=50)])

# Raw distances, before any relevance conversion (lower = closer, L2)
print("with_score   :", [(d.metadata["page"], round(float(s), 3))
                          for d, s in vector_db.similarity_search_with_score(query, k=3)])

page 2: 3.2 Attention
An attention function can be described as mapping a query and a set of key-v...
page 2: itself. To facilitate these residual connections, all sub-layers in the model, as well as ...
page 2: Figure 1: The Transformer - model architecture.
The Transformer follows this overall archi...

pages 1 or 3 : [1, 3, 1]
pages >= 5   : [12, 5, 13]
with_score   : [(2, 0.759), (12, 0.837), (2, 0.988)]


### 5. Distance strategy (an index setting, not a search type)

Cosine similarity often gets listed alongside "similarity" and "MMR" as though it were a third search mode. It is not - it is the **distance function the index uses**, fixed when the index is built. `FAISS.from_documents` takes `distance_strategy`, and `DistanceStrategy` offers `EUCLIDEAN_DISTANCE` (the default), `MAX_INNER_PRODUCT`, `COSINE`, `DOT_PRODUCT` and `JACCARD`.

Two things are worth knowing before you reach for `COSINE` here:

1. `all-MiniLM-L6-v2` returns unit-normalised vectors, and for unit vectors L2 distance and cosine are monotonically related. **The ranking is identical whichever you pick** - run the cell to confirm.
2. `langchain_community`'s FAISS wrapper only builds a special index for `MAX_INNER_PRODUCT` (`IndexFlatIP`); everything else, `COSINE` included, silently falls back to `IndexFlatL2`. But `COSINE` *does* switch the relevance-score conversion to `1 - distance`, which is the formula for cosine *distance* and simply wrong for an L2 one. The result is a working search with misleading scores. `MAX_INNER_PRODUCT` has its own quirk: the conversion inverts, so the best match gets the lowest relevance score.

The practical advice for this notebook: leave FAISS on its default `EUCLIDEAN_DISTANCE`, and when you want scores in a clean 0-1 range, use Chroma or Pinecone with a cosine metric instead.

In [14]:
from langchain_community.vectorstores.utils import DistanceStrategy

for label, kwargs in (
    ("EUCLIDEAN (default)", {}),
    ("COSINE", {"distance_strategy": DistanceStrategy.COSINE}),
    ("MAX_INNER_PRODUCT", {"distance_strategy": DistanceStrategy.MAX_INNER_PRODUCT}),
):
    store = FAISS.from_documents(split_docs, embeddings, **kwargs)
    pages = [d.metadata["page"] for d in store.similarity_search(query, k=3)]
    scores = [round(float(s), 3) for _, s in store.similarity_search_with_relevance_scores(query, k=3)]
    print(f"{label:20} index={type(store.index).__name__:13} pages={pages} relevance={scores}")

print()
print("Identical pages, different score scales - and note COSINE built an IndexFlatL2,")
print("while MAX_INNER_PRODUCT scores the best match lowest.")

EUCLIDEAN (default)  index=IndexFlatL2   pages=[2, 12, 2] relevance=[0.463, 0.408, 0.301]
COSINE               index=IndexFlatL2   pages=[2, 12, 2] relevance=[0.241, 0.163, 0.012]
MAX_INNER_PRODUCT    index=IndexFlatIP   pages=[2, 12, 2] relevance=[0.379, 0.419, 0.494]

Identical pages, different score scales - and note COSINE built an IndexFlatL2,
while MAX_INNER_PRODUCT scores the best match lowest.


### 6. Keyword search (BM25)

Everything above is **dense** retrieval: the query and the chunks become vectors, and closeness in vector space stands in for meaning. That fails on the things embeddings smooth away - exact product codes, error numbers, rare proper nouns, a specific acronym. If you ask for `BLEU` and the model has never seen it, no amount of `k` will help.

**BM25** is the classic **sparse**, lexical alternative: it scores on term overlap, weighting rare terms higher and long documents lower. No embedding model and no vector store involved - it runs over the chunk list directly.

Requires `pip install rank_bm25`.

In [15]:
from langchain_community.retrievers import BM25Retriever

# BM25 indexes the raw text - no embeddings, no vector store
bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 3

for i, doc in enumerate(bm25_retriever.invoke(query)):
    print(f"--- BM25 result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:200])
    print()

--- BM25 result 1 (page 5) ---
One is the total computational complexity per layer. Another is the amount of computation that can
be parallelized, as measured by the minimum number of sequential operations required.
The third is th

--- BM25 result 2 (page 2) ---
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, 

--- BM25 result 3 (page 5) ---
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types.n is the sequence length,d is the representation dimension,k is the kernel
siz



### 7. Hybrid search (dense + sparse)

Hybrid search runs a dense retriever and a sparse one over the same corpus and merges the two ranked lists. It catches both the paraphrase the keyword search misses and the exact term the embedding blurs.

`EnsembleRetriever` merges with **Reciprocal Rank Fusion**: each document scores `sum(weight / (60 + rank))` across the lists it appears in, so a chunk that both retrievers rank highly wins, and the two score scales - which are not comparable - never have to be reconciled.

Note it returns the *union* of the result sets, so asking two retrievers for 3 documents each can yield up to 6. Set `k` on the individual retrievers to control the size.

On LangChain 1.x this lives in `langchain_classic.retrievers`, **not** `langchain.retrievers` - most tutorials online still show the old path, which no longer exists.

In [16]:
from langchain_classic.retrievers import EnsembleRetriever

dense_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6],  # must sum to 1.0; tilt towards whichever suits your corpus
)

print("keyword only:", [d.metadata.get("page") for d in bm25_retriever.invoke(query)])
print("dense only  :", [d.metadata.get("page") for d in dense_retriever.invoke(query)])
print("hybrid      :", [d.metadata.get("page") for d in hybrid_retriever.invoke(query)])
print()

for i, doc in enumerate(hybrid_retriever.invoke(query)):
    print(f"--- Hybrid result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:200])
    print()

keyword only: [5, 2, 5]
dense only  : [2, 12, 2]
hybrid      : [2, 12, 2, 5, 2, 5]

--- Hybrid result 1 (page 2) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as 

--- Hybrid result 2 (page 12) ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS

--- Hybrid result 3 (page 2) ---
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a sta

--- Hybrid result 4 (page 5) ---
One is the total computational complexity per layer. Another is the amount of computation that can
be parallelized, as measured by the minimum number of sequential operations re

### 8. Reranking (cross-encoder)

Retrieval embeds the query and the chunk *separately* - the model never sees them together, which is what makes it fast enough to index thousands of chunks in advance. A **cross-encoder** does the opposite: it reads the query and one chunk as a single input and scores the pair directly. Far more accurate, far too slow to run over the whole corpus.

So they get combined: retrieve a wide net with the bi-encoder (`k=20`), then rerank and keep the best few. This is usually the largest quality gain per line of code in a RAG pipeline.

`ContextualCompressionRetriever` is the generic wrapper - a base retriever plus a compressor. `CrossEncoderReranker` is one compressor; others trim or summarise the chunks instead.

In [17]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Downloads a small reranking model (~90 MB) on first run
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

# Cast a wide net, then let the cross-encoder pick the best 3
wide_retriever = vector_db.as_retriever(search_kwargs={"k": 20})
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=3),
    base_retriever=wide_retriever,
)

print("before rerank:", [d.metadata.get("page") for d in wide_retriever.invoke(query)][:6], "...")
print("after rerank :", [d.metadata.get("page") for d in rerank_retriever.invoke(query)])
print()

for i, doc in enumerate(rerank_retriever.invoke(query)):
    print(f"--- Reranked {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:200])
    print()

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6585.94it/s]


before rerank: [2, 12, 2, 4, 5, 13] ...
after rerank : [1, 1, 12]

--- Reranked 1 (page 1) ---
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is


--- Reranked 2 (page 1) ---
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved


--- Reranked 3 (page 12) ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS



### 9. Score and by-vector variants of the search API

`similarity_search` returns bare `Document`s. FAISS exposes three more variants of the same search, and the distinction between the two kinds of "score" catches people out:

| Method | Returns | Score meaning |
|---|---|---|
| `similarity_search` | `[Document]` | - |
| `similarity_search_with_score` | `[(Document, float)]` | **Raw FAISS distance. Lower is closer.** |
| `similarity_search_with_relevance_scores` | `[(Document, float)]` | Converted to 0-1. **Higher is closer.** |
| `similarity_search_by_vector` | `[Document]` | - |
| `similarity_search_with_score_by_vector` | `[(Document, float)]` | Raw distance again. |

So the two score methods sort in **opposite directions**. `with_score` hands you the untouched L2 distance from the index; `with_relevance_scores` runs that distance through the conversion discussed in section 5, which is what `similarity_score_threshold` filters on.

The `_by_vector` forms skip the embedding step and take a vector you already have. That is useful when you want to embed a query once and reuse it, when you are searching with a vector that came from somewhere else, or for "more like this" - feeding an existing chunk's own embedding back in as the query. There is a `max_marginal_relevance_search_by_vector` too.

In [ ]:
# Raw distance: LOWER is closer
print("similarity_search_with_score  (raw L2 distance, lower = closer)")
for doc, score in vector_db.similarity_search_with_score(query, k=3):
    print(f"   distance {score:.4f}   page {doc.metadata.get('page')}   {doc.page_content[:60]}...")

# Converted relevance: HIGHER is closer. Same documents, opposite ordering of the number.
print()
print("similarity_search_with_relevance_scores  (0-1 scale, higher = closer)")
for doc, score in vector_db.similarity_search_with_relevance_scores(query, k=3):
    print(f"   relevance {score:.4f}  page {doc.metadata.get('page')}")

In [ ]:
# Embed the query once, then search with the vector directly
query_vector = embeddings.embed_query(query)

print("by vector:", [d.metadata.get("page") for d in vector_db.similarity_search_by_vector(query_vector, k=3)])
print("by text  :", [d.metadata.get("page") for d in vector_db.similarity_search(query, k=3)])
print("-> identical; similarity_search just calls embed_query for you")

print()
for doc, score in vector_db.similarity_search_with_score_by_vector(query_vector, k=3):
    print(f"   distance {score:.4f}   page {doc.metadata.get('page')}")

# "More like this": use an existing chunk's own embedding as the query.
# The chunk itself comes back first, at distance ~0.
seed_doc = split_docs[10]
seed_vector = embeddings.embed_query(seed_doc.page_content)

print()
print(f"seed chunk (page {seed_doc.metadata.get('page')}): {seed_doc.page_content[:70]}...")
for doc, score in vector_db.similarity_search_with_score_by_vector(seed_vector, k=3):
    print(f"   distance {score:.4f}   page {doc.metadata.get('page')}   {doc.page_content[:60]}...")

### 10. Saving and loading the index

FAISS runs in this process and holds everything in memory, so an index disappears when the kernel stops - unlike Chroma, which writes to `persist_directory` as you go, or Pinecone, which lives on a server. Re-embedding 52 chunks is quick, but re-embedding a real corpus is not, so you save the index and load it back.

`save_local(folder_path)` writes two files into the folder:

- **`index.faiss`** - the FAISS index itself, the raw vectors.
- **`index.pkl`** - the docstore and the index-position-to-document-id mapping, **pickled**.

That second file is why `load_local` refuses to run unless you pass `allow_dangerous_deserialization=True`. Unpickling executes whatever is in the file, so loading an index someone else gave you is equivalent to running their code. Setting the flag is fine for an index you saved yourself; never set it for a file you did not create.

You must pass the same embedding model on load. Nothing checks this - the saved vectors carry no record of which model produced them, so loading with a different model gives you silently meaningless results.

In [18]:
import os

FAISS_PATH = "faiss_attention_index"

vector_db.save_local(FAISS_PATH)

print("saved to:", os.path.abspath(FAISS_PATH))
for name in sorted(os.listdir(FAISS_PATH)):
    size_kb = os.path.getsize(os.path.join(FAISS_PATH, name)) / 1024
    print(f"   {name:12} {size_kb:8.1f} KB")

saved to: /Users/bk/Documents/Code/AgenticAI_Bootcamp/LangChain-Tutorial/01_Langchain/faiss_attention_index
   index.faiss      78.0 KB
   index.pkl        53.0 KB


In [19]:
# Load it back. The embedding model must match the one used to build it.
loaded_db = FAISS.load_local(
    FAISS_PATH,
    embeddings,
    allow_dangerous_deserialization=True,  # only because we wrote this file ourselves
)

print("vectors in the loaded index:", loaded_db.index.ntotal)
print("documents in the docstore  :", len(loaded_db.docstore._dict))
print()

# Searching it gives the same results as the original in-memory store
print("original:", [d.metadata.get("page") for d in vector_db.similarity_search(query, k=3)])
print("loaded  :", [d.metadata.get("page") for d in loaded_db.similarity_search(query, k=3)])

vectors in the loaded index: 52
documents in the docstore  : 52

original: [2, 12, 2]
loaded  : [2, 12, 2]


In [20]:
# Read documents straight out of the loaded index, with no search at all.
# index_to_docstore_id maps a FAISS row number to a docstore key.
print("first 3 stored documents\n")
for position in range(3):
    doc_id = loaded_db.index_to_docstore_id[position]
    doc = loaded_db.docstore.search(doc_id)
    print(f"--- row {position} | id {doc_id[:8]}... | page {doc.metadata.get('page')} ---")
    print(doc.page_content[:200])
    print()

# And the stored vector for a row, if you need the raw embedding back
vector = loaded_db.index.reconstruct(0)
print("row 0 vector:", vector.shape, vector[:5], "...")

first 3 stored documents

--- row 0 | id 3f6dd5c5... | page 0 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need


--- row 1 | id df4e187f... | page 0 ---
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine transla

--- row 2 | id 8bc9f809... | page 0 ---
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.
∗Eq

row 0 vector: (384,) [-0.11447725 -0.12833664  0.03361825 -0.01419523  0.04273785] ...


### Where each technique fits

| Technique | Layer | Use it when |
|---|---|---|
| `similarity` | search type | Default; you want the `k` closest chunks. |
| `mmr` | search type | Top hits are near-duplicates and waste prompt space. |
| `similarity_score_threshold` | search type | Off-topic questions must return nothing rather than noise. |
| Metadata filter | query option | You can narrow by page, source, date, tenant, permissions. |
| Distance metric | index setting | Fixed when the index is built; affects the score scale. |
| BM25 | separate retriever | Exact terms, codes, rare names the embedding blurs. |
| Hybrid / RRF | composition | Production RAG defaults - you want both of the above. |
| Cross-encoder rerank | post-processing | Precision matters and you can afford one extra model pass. |

Still further up the stack, and all in `langchain_classic.retrievers`: `MultiQueryRetriever` (an LLM rewrites the query several ways and unions the hits), `ParentDocumentRetriever` (embed small chunks, return their larger parent), `SelfQueryRetriever` (an LLM turns "papers after 2020 about attention" into a metadata filter), and `MultiVectorRetriever` (index summaries or hypothetical questions, return the source chunk). Each needs an LLM or extra indexing, so they are out of scope here.